In [ ]:
# Loading and Identifying NetCDF Files:
# - Uses the os module to access a specified directory and identify all files with the .nc extension.
# - Iterates through the directory to process each NetCDF file individually.
# Defining the Geographic Study Area:
# - Specifies the geographical boundaries of Tehran:
 # - Latitude: 35.57–35.83°N.
 # - Longitude: 51.12–51.61°E.
# Extracting Satellite Variables:
# - Uses the netCDF4 library to open each NetCDF file and access its PRODUCT group.
# - Extracts latitude, longitude, and nitrogendioxide_tropospheric_column (tropospheric NO₂ vertical column density).
# Filtering Data over Tehran:
# - Flattens the extracted arrays into one-dimensional arrays.
# - Iterates through the satellite observations and retains only those located within the defined Tehran boundaries.
# - Stores the selected latitude, longitude, and NO₂ values in a dictionary.
# Exporting the Extracted Data:
# - Converts the filtered observations into a Pandas DataFrame.
# - Saves each file's Tehran-specific observations as a separate CSV file, using the original filename with the suffix _Tehran.
# - Prints a confirmation message after successfully saving each CSV file.

In [ ]:
import pandas as pd
from netCDF4 import Dataset
import os

# Path to the folder containing NetCDF files
folder_path = "C:/Users/intel/Documents/SENTINEL-5P/7"

# Geographical Scope of Tehran
tehran_lat_range = (35.57, 35.83)  # Latitude
tehran_lon_range = (51.12, 51.61)  # Longitude

# Listing all NetCDF files in the folder
for file_name in os.listdir(folder_path):
    if file_name.endswith(".nc"):  # Process only files with the .nc extension
        file_path = os.path.join(folder_path, file_name)

        # Extracting the input filename (without extension)
        base_name = os.path.splitext(file_name)[0]

        # Opening a NetCDF file
        with Dataset(file_path, mode='r') as nc_file:
            product_group = nc_file.groups['PRODUCT']  # Access to the PRODUCT group
            latitudes = product_group.variables['latitude'][:]  # Reading Latitudes
            longitudes = product_group.variables['longitude'][:]  # Reading Longitudes
            no2_values = product_group.variables['nitrogendioxide_tropospheric_column'][:]  # Reading NO2 Values

        # Converting Data into One-Dimensional Arrays
        latitudes_flat = latitudes.flatten()
        longitudes_flat = longitudes.flatten()
        no2_values_flat = no2_values.flatten()

        # Filtering data based on the Tehran area
        tehran_data = {
            'Latitude': [],
            'Longitude': [],
            'NO2': []
        }

        for lat, lon, no2 in zip(latitudes_flat, longitudes_flat, no2_values_flat):
            if (tehran_lat_range[0] <= lat <= tehran_lat_range[1]) and (tehran_lon_range[0] <= lon <= tehran_lon_range[1]):
                tehran_data['Latitude'].append(lat)
                tehran_data['Longitude'].append(lon)
                tehran_data['NO2'].append(no2)

        # Convert to DataFrame
        df = pd.DataFrame(tehran_data)

        # Output filename (same as the input filename, with the suffix _Tehran)
        output_file_name = f"{base_name}_Tehran.csv"

        # Save to CSV file
        df.to_csv(output_file_name, index=False)
        print(f"Data related to Tehran from the file {file_name} were saved in the file {output_file_name}.")